In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
from typing import List, Tuple, Callable, Union, Dict, Any, Optional
from collections import defaultdict
from numpy.polynomial.legendre import leggauss
from pyomo.environ import *
from scipy.optimize import linprog
from sympy.core.relational import Relational as SympyRelational
import itertools as itools
from sympy.logic.boolalg import BooleanTrue, BooleanFalse
import numpy as np
import chaospy as cp
import math
import pickle
from pathlib import Path
import sympy as sp
import time
import pandas as pd
import pyomo.environ as pyo
from tqdm.auto import tqdm   # auto-detects notebook vs terminal
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import itertools
from functools import reduce
from gurobipy import nlfunc
import operator

## Smolyak quadrature

In [2]:
def smolyak_nodes_weights(
        n_theta: int,
        level: int,
        rule: str = "gaussian",
        growth: bool = True,
):
    """
    Returns:
        ns_u: (N, n_theta) array of Smolyak nodes in u-space (each u_k in [-1,1])
        ws_u: (N,) array of weights for integrating over [-1,1]^n_theta
                   i.e., sum_i ws_u[i] * f(ns_u[i]) ≈ ∫_{[-1,1]^n} f(u) du
    """
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    # Chaospy returns nodes shape (n_theta, N) and weights for expectation
    nodes, wE = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )

    nodes = np.asarray(nodes, dtype=float)  # (n_theta, N)
    wE = np.asarray(wE, dtype=float).ravel()  # (N,)

    ns_u = nodes.T  # (N, n_theta)

    # Convert expectation weights to integral weights over [-1,1]^n:
    # E[f(U)] = ∫ f(u) p(u) du with p(u)=1/2^n on [-1,1]^n
    # => ∫ f(u) du = 2^n * E[f(U)]
    ws_u = (2.0 ** n_theta) * wE

    return ns_u, ws_u


def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        theta_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[1]
        return float(theta_min), float(theta_max)

    try:
        region = solution.get_region(theta_vector.reshape(-1, 1))
        coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
        max_coefficients = coefficients[max_idx]
        min_coefficients = coefficients[min_idx]
        # theta_max = float(max_coefficients @ theta_vector_aug)
        # theta_min = float(min_coefficients @ theta_vector_aug)
        theta_max = (max_coefficients @ theta_vector_aug).item()
        theta_min = (min_coefficients @ theta_vector_aug).item()
        return theta_min, theta_max
    except:
        raise ValueError(
            "The provided theta_vector is not inside any critical region of the solution.")


def map_u_to_theta_and_jacobian(solutions, u, dvector):
    """
    Given stage-aligned parametric solutions for theta_k and the current
    disturbance vector d, return the theta vector and the Jacobian.

    Parameters
    ----------
    solutions : list
        Stage-aligned list. Each entry is either:
        - a valid parametric solution object for stage k, or
        - None for an empty / unusable stage.
    u : np.ndarray
        1D array of canonical coordinates.
    dvector : np.ndarray
        Current disturbance/design vector.

    Returns
    -------
    theta_values : np.ndarray
        Stage-aligned theta vector, same length as solutions.
    jacobian : float
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if sol is None:
            theta_k = 0.0
            theta_values.append(theta_k)
            continue

        if isinstance(dvector, np.ndarray):
            thetavector = np.block([np.array(theta_values, dtype=float), dvector])
        else:
            thetavector = np.array(theta_values, dtype=float)

        thetamin, thetamax = theta_interval_at_point(sol, thetavector)
        length = thetamax - thetamin

        theta_k = 0.5 * length * u[k] + 0.5 * (thetamax + thetamin)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype=float), jacobian


def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float],
                                      d_vector: np.ndarray = None, rule: str = "gaussian", ns_u=None,
                                      ws_u=None) -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callaxble f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)

    if ns_u is None or ws_u is None:
        ns_u, ws_u = smolyak_nodes_weights(
            n_theta=n_theta,
            level=level,
            rule=rule,
            growth=True,
        )

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(ns_u.shape[0]):
        u_vector = ns_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(
            solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
        stochastic_flexibility += func_value * jacobian * ws_u[i]

    end = time.time()
    print(
        f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility

In [4]:
def mpformulate_theta_bounds(flex_sol, num_theta: int, theta_bounds: list, num_design: int = 0, design_bounds: list = None, psi_idx: int = 0, theta_m: int = 0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty(
        (len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx, :num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")

    c = np.hstack([np.array([-1, 1]).reshape(1, -1),
                  np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1, 1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')

    row1_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1),
                          np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -
                  np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')

    x_lb = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1, 1)), -
                  x_lb.reshape(-1, 1), x_ub.reshape(-1, 1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')

    if F0.size == 0 and theta_m == 0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([])

    F = np.vstack([F0, F0, np.zeros((1, num_design)), np.zeros(
        (4*(num_theta-theta_m), num_design))]) if num_design > 0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]
                      ) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')

    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')

    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')

    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list)
                                                                         else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list)
                                                                        else [])).reshape(-1, 1)

    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')

    return A, b, c, H, A_t, b_t, F

In [5]:
def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None, mp_algo: mpqp_algorithm = mpqp_algorithm.combinatorial):
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)

    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
            flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        # print(f'A.shape:{A.shape}')
        # print(f'b.shape: {b.shape}')
        # print(f'F.shape: {F.shape}')
        if F.size != 0:
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("error", category=UserWarning, message="The chebychev ball has either a radius of zero, or the problem is not feasible!")
                    prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)

                prob.process_constraints()
                solution = solve_mpqp(problem=prob, algorithm=mp_algo)

                prob_dict[f"t{i}"] = prob
                theta_bound_dict[f"t{i}"] = solution

            except UserWarning as w:
                # MPLP infeasible / degenerate
                print(f"[theta {i}] MPLP infeasible / zero Chebyshev ball: {w}")
                prob_dict[f"t{i}"] = None
                theta_bound_dict[f"t{i}"] = None

        else:
            # LP fallback branch via scipy.optimize.linprog
            linres = linprog(c=c, A_ub=A, b_ub=b)

            if not linres.success:
                print(f"[theta {i}] linprog failed: status={linres.status}, "f"message={linres.message}")
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = None
            else:
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = [linres.x[1], linres.x[0]]

        print(f"Finished solving for theta{i+1}")

    probs = [p for _, p in prob_dict.items()]
    sols = [sol for _, sol in theta_bound_dict.items()]
    return probs, sols

In [6]:
def create_discrete_state_list(ny:int , states: list = None, unallowed_states: list[tuple] = None) -> list:
    
    states = states or [[0, 1] for _ in range(ny)]
    unallowed_set = set(unallowed_states or ())
    
    if isinstance(states[0], list) and len(states) != ny:
        ValueError(f"{len(states)} != {ny}. Number of lists within states should equal ny")
    
    return [s for s in itools.product(*states) if s not in unallowed_set]

In [7]:
def joint_state_prob_dict(
    dicts: list[dict],
    unallowed_states: list[tuple] | None = None,
    renormalize: bool = False,
    check_probs: bool = True,
    tol: float = 1e-12,
) -> dict[tuple, float]:
    """
    Parameters
    ----------
    dicts : list[dict]
        Each dict maps state_value -> probability for one variable.
    unallowed_states : list[tuple] | None
        State tuples to exclude, e.g. [(0,1,1), (1,0,0)].
    renormalize : bool
        If True, renormalize remaining probabilities to sum to 1
        after removing unallowed states.
    check_probs : bool
        If True, verify each input dictionary sums to 1.
    tol : float
        Tolerance used for probability-sum checks.

    Returns
    -------
    dict[tuple, float]
        Mapping from state tuple -> probability
    """
    if not dicts:
        return {}

    unallowed_set = set(unallowed_states or ())

    if check_probs:
        for i, d in enumerate(dicts):
            s = sum(d.values())
            if not math.isclose(s, 1.0, rel_tol=0.0, abs_tol=tol):
                raise ValueError(f"dicts[{i}] probabilities sum to {s}, not 1")

    vp_lists = [list(d.items()) for d in dicts]

    result = {}
    for combo in itools.product(*vp_lists):
        state = tuple(v for v, _ in combo)

        if state in unallowed_set:
            continue

        prob = 1.0
        for _, p in combo:
            prob *= p

        result[state] = prob

    if renormalize:
        total = sum(result.values())
        if math.isclose(total, 0.0, rel_tol=0.0, abs_tol=tol):
            raise ValueError("No probability mass remains after removing unallowed states")
        result = {state: p / total for state, p in result.items()}

    return result

In [8]:
suppliers = ['A', 'B', 'C']
markets = ['D', 'E', 'F']
sites = ['0', '1', '2', '3']

supply_theta_nominal = {'A': 45000, 'B': 100000, 'C': 75000}
supply_theta_stddev = {'A': np.sqrt(7500), 'B': np.sqrt(16667), 'C': np.sqrt(12500)}

conv = 1/2.4 # basis is feed

demand = {'D': 0, 'E': 60000, 'F': 25000} # demand of product

no_trips = 365 # number of trips made by each truck per year, basically one round trip per day per truck

site_cap = {'0': 50000, '1': 50000, '2': 50000, '3': 75000} # capacity of each site, basis is feed

capex_factor = {'0': 5000.00001, '1': 5000.000011, '2': 5000.000012, '3': 5000.000013}

availability_factor = 0.959260199

t_bounds = [(supply_theta_nominal['A'] - 4*supply_theta_stddev['A'], supply_theta_nominal['A'] + 4*supply_theta_stddev['A']),
            (supply_theta_nominal['B'] - 4*supply_theta_stddev['B'], supply_theta_nominal['B'] + 4*supply_theta_stddev['B']),
            (supply_theta_nominal['C'] - 4*supply_theta_stddev['C'], supply_theta_nominal['C'] + 4*supply_theta_stddev['C'])]

d_bounds = [(0, 1000), (0, 1000), (0, 1000), (0, 1000)]
nt = len(t_bounds)
nd = len(d_bounds)
disruption_dict = {
    'Site 1': {
        1: 0.8,
        0.5: 0.15,
        0: 0.05
    },
    'Site 2':{
        1: 0.7,
        0.5: 0.2,
        0: 0.1
    },
    'Site 3':{
        1: 0.6,
        0.5: 0.25,
        0: 0.15
    },
    'Site 4':{
        1: 0.9,
        0.5: 0.05,
        0: 0.05
    }
}

y_dict = dict()
combo_idx = list(disruption_dict.keys())
combo_lists = [tuple(disruption_dict[c_idx].keys()) for c_idx in combo_idx]

for combo in itertools.product(*combo_lists):
    probs = [disruption_dict[c_idx][c] for c_idx, c in zip(combo_idx, combo)]
    y_dict[combo] = reduce(operator.mul, probs, 1)

y_state_list = list(y_dict.keys())

case_study = 'NJC2025_Ex3_2'
quad_method = 'smolyak'
smolyak_level = 16

feasibility_algo = mpqp_algorithm.geometric
theta_bounds_algo = mpqp_algorithm.geometric

feas_algo_name = 'gp' if feasibility_algo.name in ['geometric_parallel', 'geometric'] else ('cb' if feasibility_algo in ['combinatorial', 'combinatorial_parallel'] else '')
tb_algo_name = 'gp' if theta_bounds_algo.name in ['geometric_parallel', 'geometric'] else ('cb' if theta_bounds_algo in ['combinatorial', 'combinatorial_parallel'] else '')

def create_flexibility_model(tbounds:list, dbounds:list, y_list: tuple = None):
    m = MPModeler()
    # print(f'y_list: {y_list}')
    
    u = m.add_var(name='u')
    product = {si: m.add_var(name=f'product[{si}]') for si in sites}
    feed = {si: m.add_var(f'feed[{si}]') for si in sites}
    feed_ship = {(su, si): m.add_var(name=f'feed_ship[{su},{si}]') for su, si in itertools.product(suppliers, sites)}
    product_ship = {(si, ma): m.add_var(name=f'product_ship[{si},{m}]') for si, ma in itertools.product(sites, markets)}
    capex = m.add_var(name='capex')
    
    supply_theta = {su: m.add_param(name=f'supply_theta[{su}]') for su in suppliers}
    trans_cap = {si: m.add_param(name=f'trans_cap[{si}]') for si in sites}
    
    m.add_constrs(sum(feed_ship[su, si] for su in suppliers) == feed[si] for si in sites)
    m.add_constr(capex == sum(capex_factor[si]*trans_cap[si] for si in sites))
    m.add_constrs(product[si] == feed[si]*conv for si in sites)
    m.add_constrs(sum(product_ship[si, ma] for ma in markets) == product[si] for si in sites)
    
    m.add_constrs(sum(feed_ship[su, si] for si in sites) <= supply_theta[su] + u for su in suppliers)
    m.add_constrs(feed[si] <= site_cap[si] * availability_factor + u for si in sites)
    m.add_constrs(demand[ma] <= sum(product_ship[si, ma] for si in sites) + u for ma in markets)
    m.add_constrs(feed[si] + product[si] <= trans_cap[si]*no_trips*y_list[int(si)] + u for si in sites)
    
    m.add_constrs(supply_theta[su] >= tbounds[su_idx][0] for su_idx, su in enumerate(suppliers))
    m.add_constrs(supply_theta[su] <= tbounds[su_idx][1] for su_idx, su in enumerate(suppliers))
    m.add_constrs(trans_cap[si] >= dbounds[si_idx][0] for si_idx, si in enumerate(sites))
    m.add_constrs(trans_cap[si] <= dbounds[si_idx][1] for si_idx, si in enumerate(sites))
    
    m.set_objective(u)
    
    return m

def joint_pdf(theta: list, eps:float = 1e-8):
    theta_a, theta_b, theta_c = theta
    a_nom, b_nom, c_nom = supply_theta_nominal['A'], supply_theta_nominal['B'], supply_theta_nominal['C']
    a_std, b_std, c_std = supply_theta_stddev['A'], supply_theta_stddev['B'], supply_theta_stddev['C']
    
    contrib_a = (1/np.sqrt(2*np.pi)) * (1/a_std) * np.exp(-(theta_a - a_nom)**2/(2 * a_std**2))
    contrib_b = (1/np.sqrt(2*np.pi)) * (1/b_std) * np.exp(-(theta_b - b_nom)**2/(2 * b_std**2))
    contrib_c = (1/np.sqrt(2*np.pi)) * (1/c_std) * np.exp(-(theta_c - c_nom)**2/(2 * c_std**2))
      
    return contrib_a * contrib_b * contrib_c

def pdf_builder(theta, n, eps=1e-8):
    theta_a = theta[(0, n)]
    theta_b = theta[(1, n)]
    theta_c = theta[(2, n)]
    
    a_nom, b_nom, c_nom = supply_theta_nominal['A'], supply_theta_nominal['B'], supply_theta_nominal['C']
    a_std, b_std, c_std = supply_theta_stddev['A'], supply_theta_stddev['B'], supply_theta_stddev['C']
    
    contrib_a = (1/np.sqrt(2*math.pi)) * (1/a_std) * nlfunc.exp(-((theta_a - a_nom) * (theta_a - a_nom))/(2 * a_std * a_std))
    contrib_b = (1/np.sqrt(2*math.pi)) * (1/b_std) * nlfunc.exp(-((theta_b - b_nom) * (theta_b - b_nom))/(2 * b_std * b_std))
    contrib_c = (1/np.sqrt(2*math.pi)) * (1/c_std) * nlfunc.exp(-((theta_c - c_nom) * (theta_c - c_nom))/(2 * c_std * c_std))
     
    return contrib_a * contrib_b * contrib_c

cost_coeff_list = [v for k,v in capex_factor.items()]
def cost_function(d):
    coeffs = np.array(cost_coeff_list)
    return float(np.dot(coeffs, d))

## SF Expression

In [9]:
def get_bounds_regions(sols: List, min_idx: int = 1, max_idx: int = 0):
    theta_bounds_list = []
    theta_regions_list = []

    for theta_sol in sols:
        if theta_sol is None:
            theta_bounds_list.append(None)
            theta_regions_list.append(None)
            continue
        
        if not getattr(theta_sol, "critical_regions", None):
            theta_bounds_list.append(None)
            theta_regions_list.append(None)
            continue
        
        min_max_list = []
        region_list = []

        for cr in theta_sol.critical_regions:
            # Store bounds
            Ab = np.concatenate([cr.A, cr.b], axis=1)[:2]
            min_max_list.append([Ab[min_idx].tolist(), Ab[max_idx].tolist()])

            # Store region constraints
            Ef = np.concatenate([cr.E, -cr.f], axis=1)
            region_array = np.array([row.tolist() for row in Ef], dtype=float)
            region_list.append(region_array)

        # Append per-theta data
        theta_bounds_list.append(np.array(min_max_list))
        # <-- each region is a 2D array
        theta_regions_list.append(np.array(region_list, dtype=object))

    return theta_bounds_list, theta_regions_list


def generate_region_combos(region_sizes, n_gl):
    """Generate region index combinations based on critical region structure."""
    n_theta = len(region_sizes)
    region_combo_shape = []
    for k in range(n_theta):
        n_paths = int(np.prod(n_gl[:k])) if k > 0 else 1
        region_combo_shape.extend([range(region_sizes[k])] * n_paths)
    return list(itools.product(*region_combo_shape))


def affine_expr(coeffs, symbols):
    return sum(c * s for c, s in zip(coeffs[:-1], symbols)) + coeffs[-1]


def normalized_lhs(ineq):
    return ineq.lhs.expand() if hasattr(ineq, 'lhs') else None

In [10]:
def _prepare_state_data(state, tbounds, dbounds, *, solve_algo, theta_algo, log=False):
    """
    Build and solve the flexibility problem for one state, then return only valid theta-related data.

    Returns
    -------
    dict with keys:
        state
        flex_sol
        sol_list
        t_bounds_list
        t_regions_list
        filtered_solutions
        filtered_theta_bounds
        filtered_theta_regions
    or None if no valid theta regions exist.
    """
    state_model = create_flexibility_model(y_list=state, tbounds=tbounds, dbounds=dbounds)
    state_prob = state_model.formulate_problem()
    state_prob.process_constraints()

    flex_sol = solve_mpqp(problem=state_prob, algorithm=solve_algo)
    
    numt = len(tbounds)
    numd = len(dbounds)

    if log:
        print(
            f"Number of critical regions in for flexibility function for state {state}: {len(flex_sol.critical_regions)}")

    _, sol_list = get_theta_bounds(flex_sol=flex_sol, numt=numt, numd=numd, tbounds=tbounds, dbounds=dbounds,
                                   mp_algo=theta_algo)

    t_bounds_list, t_regions_list = get_bounds_regions(sols=sol_list)

    filtered_solutions = []
    filtered_theta_bounds = []
    filtered_theta_regions = []

    for k, (sol, tb, tr) in enumerate(zip(sol_list, t_bounds_list, t_regions_list)):
        if sol is None or tb is None or tr is None:
            # mark as empty stage but preserve position
            filtered_solutions.append(None)
            filtered_theta_bounds.append(None)
            filtered_theta_regions.append(None)
        else:
            filtered_solutions.append(sol)
            filtered_theta_bounds.append(tb)
            filtered_theta_regions.append(tr)

    return {
        "state": state,
        "flex_sol": flex_sol,
        "sol_list": sol_list,
        "theta_bounds_list": t_bounds_list,
        "theta_regions_list": t_regions_list,
        "filtered_solutions": filtered_solutions,
        "filtered_theta_bounds": filtered_theta_bounds,
        "filtered_theta_regions": filtered_theta_regions,
    }

In [11]:
def _safe_sf_call(func, state, label, **kwargs):
    try:
        return func(**kwargs)
    except ValueError as e:
        if "not inside any critical region" in str(e):
            print(f"Skipping {label} SF for state {state}: {e}")
            return None
        raise

In [12]:
def calculate_sm_esf(y_d: dict, prepared_data_by_s: dict, d_v, s_level: int, ns_u=None, ws_u=None):
    """
    Smolyak ESF using stage-aligned prepared data.
    Assumes map_u_to_theta_and_jacobian() now handles None entries in solutions.
    """
    sm_esf = 0.0
    esf_by_state = {}

    for s, p in y_d.items():
        try:
            data = prepared_data_by_s.get(s)

            if data is None:
                print(f"No valid theta regions for state {s}; skipping state")
                continue

            sols = data.get("filtered_solutions", None)

            if sols is None or len(sols) == 0:
                print(f"No filtered solutions for state {s}; skipping state")
                continue

            if not any(sol is not None for sol in sols):
                print(f"All filtered solutions are None for state {s}; skipping state")
                continue

            sf_idx_smolyak = _safe_sf_call(
                calculate_stocflexibility_smolyak,
                state=s,
                label="Smolyak",
                solutions=sols,
                level=s_level,
                joint_func=joint_pdf,
                d_vector=d_v,
                ns_u=ns_u,
                ws_u=ws_u,
            )

            if sf_idx_smolyak is not None:
                esf_by_state[s] = sf_idx_smolyak
                sm_esf += sf_idx_smolyak * p

        except ValueError as e:
            print(f"Skipping state {s} due to ValueError: {e}")
            continue

        print(f"Finished for state {s}.")

    return sm_esf, esf_by_state

In [13]:
# prepared_data_by_state = {
#     state: _prepare_state_data(
#         state,
#         tbounds=t_bounds,
#         dbounds=d_bounds,
#         solve_algo=feasibility_algo,
#         theta_algo=theta_bounds_algo,
#     )
#     for state in y_dict
# }
# 
# # load_prepared_data_by_state = prepared_data_by_state

In [14]:
# with open(f'state_data_{case_study}.pkl', 'wb') as f:
#     pickle.dump(prepared_data_by_state, f)

In [15]:
with open(f'state_data_{case_study}.pkl', 'rb') as f:
    load_prepared_data_by_state = pickle.load(f)

In [16]:
load_prepared_data_by_state[1,1,1,1].keys()

dict_keys(['state', 'flex_sol', 'sol_list', 'theta_bounds_list', 'theta_regions_list', 'filtered_solutions', 'filtered_theta_bounds', 'filtered_theta_regions'])

In [17]:
len(load_prepared_data_by_state[1,1,1,1]['flex_sol'])

16

In [21]:
design_vector = np.array([1000,1000,1000,1000])

In [22]:
esf_sm, sf_by_state_sm = calculate_sm_esf(y_d=y_dict, prepared_data_by_s=load_prepared_data_by_state, d_v=design_vector, s_level=smolyak_level)

Smolyak stochastic flexibility computed in 3.5715 seconds.
Finished for state (1, 1, 1, 1).
Smolyak stochastic flexibility computed in 3.5409 seconds.
Finished for state (1, 1, 1, 0.5).
All filtered solutions are None for state (1, 1, 1, 0); skipping state
Smolyak stochastic flexibility computed in 3.5510 seconds.
Finished for state (1, 1, 0.5, 1).
Smolyak stochastic flexibility computed in 3.6547 seconds.
Finished for state (1, 1, 0.5, 0.5).
All filtered solutions are None for state (1, 1, 0.5, 0); skipping state
All filtered solutions are None for state (1, 1, 0, 1); skipping state
All filtered solutions are None for state (1, 1, 0, 0.5); skipping state
All filtered solutions are None for state (1, 1, 0, 0); skipping state
Smolyak stochastic flexibility computed in 3.6048 seconds.
Finished for state (1, 0.5, 1, 1).
Smolyak stochastic flexibility computed in 3.7131 seconds.
Finished for state (1, 0.5, 1, 0.5).
All filtered solutions are None for state (1, 0.5, 1, 0); skipping state
Sm

In [23]:
esf_sm

0.6908729888717088

In [24]:
sf_by_state_sm

{(1, 1, 1, 1): 1.000666976440474,
 (1, 1, 1, 0.5): 1.000666976440474,
 (1, 1, 0.5, 1): 1.000666976440474,
 (1, 1, 0.5, 0.5): 1.000666976440474,
 (1, 0.5, 1, 1): 1.000666976440474,
 (1, 0.5, 1, 0.5): 1.000666976440474,
 (1, 0.5, 0.5, 1): 1.000666976440474,
 (1, 0.5, 0.5, 0.5): 1.000666976440474,
 (0.5, 1, 1, 1): 1.000666976440474,
 (0.5, 1, 1, 0.5): 1.000666976440474,
 (0.5, 1, 0.5, 1): 1.000666976440474,
 (0.5, 1, 0.5, 0.5): 1.000666976440474,
 (0.5, 0.5, 1, 1): 1.000666976440474,
 (0.5, 0.5, 1, 0.5): 1.000666976440474,
 (0.5, 0.5, 0.5, 1): 1.000666976440474,
 (0.5, 0.5, 0.5, 0.5): 1.000666976440474}

## Pyomo Model

In [25]:
def preprocess_sf_expressions(
    expr_list: List[sp.Expr],
    region_list: List[List[sp.Expr]],
) -> Tuple[List[Callable], List[List[Tuple[str, Callable, Callable]]], List[str]]:
    """
    Preprocess symbolic SF and region expressions into callable Pyomo functions.

    Returns:
        sf_pyomo_funcs:   list of callables f(var_dict)->Pyomo expression
        region_pyomo_funcs: list (per SF expr) of [(rel_op, lhs_func, rhs_func), ...]
        var_names:        sorted list of variable names used
    """
    assert len(expr_list) == len(
        region_list), "Mismatch between expressions and regions"

    sf_pyomo_funcs = []
    region_pyomo_funcs = []
    all_syms = set()

    # collect symbols + validate region expressions
    for sf_expr, reg_exprs in zip(expr_list, region_list):
        all_syms.update(sf_expr.free_symbols)
        for reg_expr in reg_exprs:
            if not isinstance(reg_expr, SympyRelational):
                raise ValueError(
                    f"Region expression {reg_expr} is not a Sympy relational (<=,>=,==,...)"
                )
            all_syms.update(reg_expr.free_symbols)

    var_names = sorted(str(s) for s in all_syms)

    # SF expression callables (Pyomo-safe exp/log/pi)
    for sf_expr in expr_list:
        sf_func = sp.lambdify(
            var_names,
            sf_expr,
            modules=[{"exp": pyo.exp, "log": pyo.log,
                      "ln": pyo.log, "pi": math.pi}, "sympy"],
        )
        sf_pyomo_funcs.append(lambda var_dict, f=sf_func: f(
            *[var_dict[v] for v in var_names]))

    # Region constraint callables (lhs/rhs algebra only)
    for reg_exprs in region_list:
        region_funcs = []
        for reg in reg_exprs:
            lhs_func = sp.lambdify(var_names, reg.lhs, modules="sympy")
            rhs_func = sp.lambdify(var_names, reg.rhs, modules="sympy")
            region_funcs.append((reg.rel_op, lhs_func, rhs_func))
        region_pyomo_funcs.append(region_funcs)

    return sf_pyomo_funcs, region_pyomo_funcs, var_names


def embed_sf_constraints_to_model(
    instance: ConcreteModel,
    sf_pyomo_funcs: List[Callable],
    region_pyomo_funcs: List[List[Tuple[str, Callable, Callable]]],
    var_names: List[str],
    bounds_dict: Dict[str, Tuple[float, float]],
    big_m: float = 1e4,
    initial_target: float = 1.0,
):
    """
    Embed SF constraints and region logic into a Pyomo model using preprocessed functions.
    """
    if not hasattr(instance, 'generated_constraints'):
        instance.generated_constraints = ConstraintList()

    if not hasattr(instance, 'region_constraints'):
        instance.region_constraints = ConstraintList()

    if not hasattr(instance, 'region_binaries'):
        instance.region_binaries = Var(
            RangeSet(len(sf_pyomo_funcs)), within=Binary)

    # Create design variables

    for var_name in var_names:
        if not hasattr(instance, var_name):
            setattr(instance, var_name, Var(bounds=bounds_dict[var_name]))

    # Create mapping for lambdified expressions
    var_dict = {name: getattr(instance, name) for name in var_names}

    # Mutable Param for target
    if not hasattr(instance, 'sf_target'):
        instance.sf_target = Param(mutable=True, initialize=initial_target)

    # Create sf_var and constraint
    if not hasattr(instance, 'sf'):
        instance.sf = Var(within=NonNegativeReals)

    # Store SF expressions
    sf_expr_pyomo_list = []
    for i, sf_func in enumerate(sf_pyomo_funcs):
        sf_expr = sf_func(var_dict)
        sf_expr_pyomo_list.append(sf_expr)
        # Big-M constraint: sf_expr >= target - M(1 - δ)
        delta = instance.region_binaries[i + 1]
        instance.generated_constraints.add(
            sf_expr >= instance.sf_target - big_m * (1 - delta))

    # Link to overall sf var
    if not hasattr(instance, 'sf_con'):
        instance.sf_con = Constraint(expr=instance.sf == sum(
            sf_expr_pyomo_list[i] * instance.region_binaries[i + 1] for i in range(len(sf_expr_pyomo_list))))

    # Region constraints
    for i, region_funcs in enumerate(region_pyomo_funcs):
        delta = instance.region_binaries[i + 1]

        for rel_op, lhs_func, rhs_func in region_funcs:
            lhs = lhs_func(*[var_dict[v] for v in var_names])
            rhs = rhs_func(*[var_dict[v] for v in var_names])

            if rel_op == '<=':
                instance.region_constraints.add(
                    lhs <= rhs + big_m * (1 - delta))

            elif rel_op == '<':
                instance.region_constraints.add(
                    lhs <= rhs - 1e-6 + big_m * (1 - delta))

            elif rel_op == '>=':
                instance.region_constraints.add(
                    lhs >= rhs - big_m * (1 - delta))

            elif rel_op == '>':
                instance.region_constraints.add(
                    lhs >= rhs + 1e-6 - big_m * (1 - delta))

            elif rel_op == '==':
                instance.region_constraints.add(
                    lhs >= rhs - big_m * (1 - delta))
                instance.region_constraints.add(
                    lhs <= rhs + big_m * (1 - delta))

            else:
                raise NotImplementedError(f"Unsupported operator: {rel_op}")

    # Exclusivity
    instance.region_exclusivity = Constraint(expr=sum(
        instance.region_binaries[i + 1] for i in range(len(sf_pyomo_funcs))) == 1)

In [26]:
def build_base_model_esf(d_bounds):
    m = pyo.ConcreteModel()

    # Design variables
    for j, (lb, ub) in enumerate(d_bounds):
        setattr(m, f"d{j}", pyo.Var(bounds=(lb, ub), initialize=0.5*(lb + ub)))

    def dvar(j):
        return getattr(m, f"d{int(j)}")
    m.dvar = dvar  # helper for easier access in pdf_expr

    # global ESF (sum of prob-weighted block SFs)
    m.ESF = pyo.Expression(initialize=0.0)

    # store number of design vars
    m.nd = len(d_bounds)

    return m


def add_sf_block(
    m,
    name,
    theta_bounds_list,
    theta_regions_list,
    ns_u,
    ws_u,
    t_bounds,
    pdf_expr,
    probability,
    big_m=1e5,
):
    N = int(ns_u.shape[0])
    nt = len(theta_bounds_list)
    nd = m.nd

    block = pyo.Block()
    setattr(m, name, block)
    block = getattr(m, name)

    block.probability = float(probability)
    block.N = pyo.RangeSet(0, N - 1)
    block.K = pyo.RangeSet(0, nt - 1)
    Rk = [theta_bounds_list[k].shape[0] for k in range(nt)]
    block.R = pyo.Set(block.K, initialize=lambda b, k: list(range(Rk[int(k)])))

    # nodes/weights
    block.nodes_u = pyo.Param(
        block.N, block.K,
        initialize=lambda b, n, k: float(ns_u[n, k]),
        mutable=False,
    )
    block.weights_u = pyo.Param(
        block.N,
        initialize=lambda b, n: float(ws_u[n]),
        mutable=False,
    )

    # theta, tmin, tmax, jac
    block.theta = pyo.Var(
        block.K, block.N,
        bounds=lambda b, k, n: t_bounds[int(k)],
        initialize=lambda b, k, n: 0.5 *
        (t_bounds[int(k)][0] + t_bounds[int(k)][1]),
    )
    block.tmin = pyo.Var(
        block.K, block.N,
        bounds=lambda b, k, n: t_bounds[int(k)],
    )
    block.tmax = pyo.Var(
        block.K, block.N,
        bounds=lambda b, k, n: t_bounds[int(k)],
    )
    block.jac = pyo.Var(block.N, within=pyo.NonNegativeReals, initialize=1.0)

    block.KRN = pyo.Set(
        dimen=3,
        initialize=[(k, r, n)
                    for k in range(nt)
                    for n in range(N)
                    for r in range(Rk[k])])
    block.z = pyo.Var(block.KRN, within=pyo.Binary)

    # affine helper
    def affine_bound_expr(b, k, n, coeff_vec):
        k = int(k)
        n = int(n)
        coeff_vec = np.asarray(coeff_vec, dtype=float).ravel()
        expr = 0
        for i in range(k):
            expr += float(coeff_vec[i]) * block.theta[i, n]
        offset = k
        for j in range(nd):
            expr += float(coeff_vec[offset + j]) * m.dvar(j)
        expr += float(coeff_vec[-1])
        return expr
    block._affine_bound_expr = affine_bound_expr

    # region exclusivity
    block.region_one = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            block.region_one.add(
                sum(block.z[(k, r, n)] for r in range(Rk[k])) == 1
            )

    # tmin/tmax big-M
    block.bound_link = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            for r in range(Rk[k]):
                tmin_coeff = theta_bounds_list[k][r][0]
                tmax_coeff = theta_bounds_list[k][r][1]

                tmin_expr = affine_bound_expr(block, k, n, tmin_coeff)
                tmax_expr = affine_bound_expr(block, k, n, tmax_coeff)

                zkrn = block.z[(k, r, n)]
                block.bound_link.add(
                    block.tmin[k, n] - tmin_expr <= big_m*(1 - zkrn))
                block.bound_link.add(
                    block.tmin[k, n] - tmin_expr >= -big_m*(1 - zkrn))
                block.bound_link.add(
                    block.tmax[k, n] - tmax_expr <= big_m*(1 - zkrn))
                block.bound_link.add(
                    block.tmax[k, n] - tmax_expr >= -big_m*(1 - zkrn))

    # region membership
    block.region_ineq = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            for r in range(Rk[k]):
                zkrn = block.z[(k, r, n)]
                rows = theta_regions_list[k][r]
                for row in rows:
                    row = np.asarray(row, dtype=float).ravel()
                    theta_count = k
                    if len(row) == (k + 1) + nd + 1:
                        theta_count = k + 1
                    elif len(row) != theta_count + nd + 1:
                        raise ValueError(
                            f"Bad region row length at k={k}, r={r}. Got {len(row)}."
                        )
                    lhs = 0
                    for i in range(theta_count):
                        lhs += float(row[i]) * block.theta[i, n]
                    for j in range(nd):
                        lhs += float(row[theta_count + j]) * m.dvar(j)
                    lhs += float(row[-1])

                    block.region_ineq.add(lhs <= big_m*(1 - zkrn))

    # theta mapping
    block.theta_map = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            ukn = block.nodes_u[n, k]
            block.theta_map.add(
                block.theta[k, n] == 0.5 *
                (block.tmax[k, n] - block.tmin[k, n]) * ukn
                + 0.5*(block.tmax[k, n] + block.tmin[k, n])
            )
            block.theta_map.add(block.tmax[k, n] >= block.tmin[k, n] + 1e-8)

    # jacobian
    block.jac_def = pyo.ConstraintList()
    for n in range(N):
        expr = 1
        for k in range(nt):
            expr *= 0.5*(block.tmax[k, n] - block.tmin[k, n])
        block.jac_def.add(block.jac[n] == expr)

    # SF for this block
    block.sf_expr = pyo.Expression(
        expr=sum(float(block.weights_u[n]) * block.jac[n] * pdf_expr(block, n)
                 for n in range(N))
    )

    # accumulate into global ESF with probability
    if m.ESF.expr is None or m.ESF.is_fixed():
        m.ESF.set_value(block.probability * block.sf_expr)
    else:
        m.ESF.set_value(m.ESF.expr + block.probability * block.sf_expr)

    return block


def set_esf_target(m, target):

    # if global target already exists, just update it
    if hasattr(m, "esf_target") and hasattr(m, "esf_con"):
        m.esf_target.set_value(float(target))
        return m

    # otherwise create Param + Constraint for global ESF
    esf_expr = m.ESF
    m.esf_target = pyo.Param(mutable=True, initialize=float(target))
    m.esf_con = pyo.Constraint(expr=esf_expr >= m.esf_target)
    return m


def warmstart_exact_smolyak_minlp(
    m: pyo.ConcreteModel,
    block_name: str,
    *,
    theta0_init: float = 9.0,    # must be > 8 (log domain)
    theta1_init: float = 7.0,
    tmin_init: float = 8.5,      # must be > 8
    tmax_init: float = 9.5,      # must be > tmin_init,
    d_init:list = None
):
    """
    Warm-start the Smolyak MINLP so BARON can safely evaluate exp/log terms.
    Targets a named SF block inside m (e.g. block_name="sf0").
    """
    blk = getattr(m, block_name)

    # ------------------------
    # 1) Design variables d_j  (still on root model, not block)
    # ------------------------
    
    
    j = 0
    while hasattr(m, f"d{j}"):
        v = getattr(m, f"d{j}")
        if not d_init:
            lb, ub = v.bounds
            v.set_value(0.5 * (lb + ub))
        else:
            v.set_value(d_init[j])
        j += 1

    # ------------------------
    # 2) Theta, tmin, tmax
    # ------------------------
    for n in blk.N:
        if (0, n) in blk.theta:
            blk.theta[0, n].set_value(theta0_init)
        if (1, n) in blk.theta:
            blk.theta[1, n].set_value(theta1_init)

        for k in blk.K:
            if (k, n) in blk.tmin:
                blk.tmin[k, n].set_value(tmin_init)
            if (k, n) in blk.tmax:
                blk.tmax[k, n].set_value(tmax_init)

        if n in blk.jac:
            blk.jac[n].set_value(
                max(1e-3, (0.5 * (tmax_init - tmin_init)) ** len(list(blk.K)))
            )

    # ------------------------
    # 3) Region binaries
    # ------------------------
    if hasattr(blk, "KRN") and hasattr(blk, "z"):
        for (k, r, n) in blk.KRN:
            blk.z[(k, r, n)].set_value(1 if r == 0 else 0)

In [27]:
def add_zero_sf_block(m, name, probability):
    block = pyo.Block()
    setattr(m, name, block)
    block = getattr(m, name)

    block.probability = float(probability)

    # SF for this block is exactly zero
    block.sf_expr = pyo.Expression(expr=0.0)

    # accumulate into global ESF
    if m.ESF.expr is None or m.ESF.is_fixed():
        m.ESF.set_value(block.probability * block.sf_expr)
    else:
        m.ESF.set_value(m.ESF.expr + block.probability * block.sf_expr)

    return block

In [28]:
def first_full_sf_block_name(m, prefix="sf", max_search=10000):
    for i in range(max_search):
        name = f"{prefix}{i}"
        blk = m.find_component(name)
        if blk is None:
            continue
        if blk.find_component("N") is not None and blk.find_component("theta") is not None:
            return name
    return None

In [29]:
m_sf_sm_exact = build_base_model_esf(d_bounds=d_bounds)

In [ ]:
nodes_u, weights_u = smolyak_nodes_weights(n_theta=nt, level=smolyak_level)

In [ ]:
for idx, state in enumerate(y_state_list):
    data = _prepare_state_data(state, solve_algo=mpqp_algorithm.geometric_parallel, theta_algo=mpqp_algorithm.combinatorial, log=True)

    if data is None:
        print(f"No valid theta regions for state {state}; skipping SF block")
        _ = add_zero_sf_block(m=m_sf_sm_exact, name=f"sf{idx}", probability=y_dict[state])
    else:
        _ = add_sf_block(m=m_sf_sm_exact, name=f"sf{idx}", theta_bounds_list=data["filtered_theta_bounds"], theta_regions_list=data["filtered_theta_regions"],
                         ns_u=nodes_u, ws_u=weights_u, t_bounds=t_bounds, pdf_expr=pdf_expr, probability=y_dict[state], big_m=1e5)

    print(f"Finished for state {state}.")